# 04 - Global Results Comparison for Sparse-View CT

This notebook implements the final global comparison stage of the sparse-view CT pipeline. According to the project specifications (**Page 2: "they should include a comparison plot within all methods"**), we evaluate and compare all implemented reconstruction paradigms on the **exact same degraded test slice**:

1. **Filtered Backprojection (FBP)**: The classical analytical baseline.
2. **Total p-Variation (TpV) Regularization**: The model-based non-convex variational method (using the Chambolle-Pock solver from scratch).
3. **Generalized UNet**: The deep learning learned post-processing method.

The notebook generates a unified quantitative comparison table (PSNR and SSIM) and the ultimate comparative visual panel containing all reconstructions and absolute error maps side-by-side.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox` if needed, and import PyTorch, NumPy, Matplotlib, and `IPPy` libraries.

In [ ]:
!pip install astra-toolbox

from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed"
UNET_WEIGHTS_PATH = PROJECT_ROOT / "weights" / "unet" / "unet_generalized.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

# Global Configuration
SEED = 42
IMAGE_SIZE = 256
DETECTOR_SIZE = 256
GEOMETRY = "parallel"
ANGLE_COUNTS = (180, 90, 60, 45)

torch.manual_seed(SEED)
np.random.seed(SEED)
device = utilities.get_device()

print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("Comparison outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Data Contract (Shared Test Slice)

Load the first test patient file and select the shared central slice to guarantee identical evaluation inputs across all methods.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# Load the first test patient
test_split = manifest["splits"]["test"]
patient_record = test_split["patients"][0]
patient_path = PROCESSED_DIR / patient_record["path"]
payload = torch.load(patient_path, map_location="cpu")

clean_images = payload["clean"].to(torch.float32)
sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

# Target Configuration for the comparison
n_views = 60
angle_key = str(n_views)

# Select the central slice
slice_idx = clean_images.shape[0] // 2

x_true = clean_images[slice_idx : slice_idx + 1].to(device)
y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)

print(f"Loaded Test Patient: {patient_id}")
print(f"Evaluating on central slice index: {slice_idx} with {n_views} views.")

## 3. Method 1: Filtered Backprojection (FBP)

Run FBP reconstruction on the shared test slice to serve as the classical baseline.

In [ ]:
# Initialize FBP Operator
K = operators.CTProjector(
    img_shape=(IMAGE_SIZE, IMAGE_SIZE),
    angles=np.linspace(0.0, np.pi, n_views),
    det_size=DETECTOR_SIZE,
    geometry=GEOMETRY,
    force_cpu=True,
)
fbp_solver = solvers.FBP(K)

print("Running FBP solver...")
x_fbp, _ = fbp_solver(y_delta, x_true=None, starting_point=None)

def normalize_finite(image: torch.Tensor) -> torch.Tensor:
    image = torch.nan_to_num(image.float(), nan=0.0, posinf=0.0, neginf=0.0)
    if torch.isclose(image.max(), image.min()):
        return torch.zeros_like(image, dtype=torch.float32)
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)

x_fbp_norm = normalize_finite(x_fbp.detach().cpu())
x_true_cpu = x_true.detach().cpu()

fbp_psnr = float(PSNR(x_fbp_norm, x_true_cpu))
fbp_ssim = float(SSIM(x_fbp_norm, x_true_cpu))

print(f"FBP PSNR: {fbp_psnr:.2f} dB | SSIM: {fbp_ssim:.4f}")

## 4. Method 2: Total p-Variation (TpV) Reconstruction

Run the model-based variational Chambolle-Pock solver from scratch (zeros) on the same test slice.

In [ ]:
# Variational Hyperparameters (Calibrated in notebook 02)
lmbda = 0.01          # Regularization weight
p = 0.35              # Sparsity parameter
maxiter = 150         # Maximum iterations

tpv_solver = solvers.ChambollePockTpVUnconstrained(K)

print("Running Chambolle-Pock TpV solver (starting from scratch)... ")
x_tpv, info = tpv_solver(
    y_delta,
    lmbda=lmbda,
    starting_point=None,  # Standard zero-image initialization
    x_true=x_true,
    maxiter=maxiter,
    tolf=1e-5,
    tolx=1e-5,
    p=p,
    verbose=False,
)

x_tpv_norm = normalize_finite(x_tpv.detach().cpu())

tpv_psnr = float(PSNR(x_tpv_norm, x_true_cpu))
tpv_ssim = float(SSIM(x_tpv_norm, x_true_cpu))

print(f"TpV PSNR: {tpv_psnr:.2f} dB | SSIM: {tpv_ssim:.4f}")

## 5. Method 3: Deep Learning (Generalized UNet)

Load the trained weights of the unified generalized UNet and run it on the FBP proxy of the same test slice.

In [ ]:
# Define UNet Architecture
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(),
        )
    def forward(self, x):
        return self.block(x)

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = block_cls(in_ch, out_ch)
    def forward(self, x):
        return self.block(self.pool(x))

class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, block_cls=DoubleConv):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=2, stride=2)
        self.block = block_cls(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([skip, x], dim=1)
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base_ch=32):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base_ch)
        self.enc2 = DownBlock(base_ch, 2 * base_ch)
        self.enc3 = DownBlock(2 * base_ch, 4 * base_ch)
        self.bottleneck = DownBlock(4 * base_ch, 8 * base_ch)
        self.dec3 = UpBlock(8 * base_ch, 4 * base_ch, 4 * base_ch)
        self.dec2 = UpBlock(4 * base_ch, 2 * base_ch, 2 * base_ch)
        self.dec1 = UpBlock(2 * base_ch, base_ch, base_ch)
        self.out_conv = nn.Conv2d(base_ch, out_ch, kernel_size=1)

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        h = self.bottleneck(s3)
        h = self.dec3(h, s3)
        h = self.dec2(h, s2)
        h = self.dec1(h, s1)
        return self.out_conv(h)

# Load trained generalized weights
unet = UNet(in_ch=1, out_ch=1, base_ch=32).to(device)
checkpoint = torch.load(UNET_WEIGHTS_PATH, map_location=device)
unet.load_state_dict(checkpoint["model_state_dict"])
unet.eval()

print("Running UNet model...")
fbp_input = x_fbp_norm.to(device)
with torch.no_grad():
    x_unet = torch.clamp(unet(fbp_input), 0.0, 1.0)

x_unet_norm = normalize_finite(x_unet.detach().cpu())

unet_psnr = float(PSNR(x_unet_norm, x_true_cpu))
unet_ssim = float(SSIM(x_unet_norm, x_true_cpu))

print(f"UNet PSNR: {unet_psnr:.2f} dB | SSIM: {unet_ssim:.4f}")

## 6. Quantitative Results Table

Display a clean ASCII comparative table summarizing the performance of all three methods.

In [ ]:
print("="*55)
print(f"{ 'METHOD':<20 } | { 'PSNR (dB)':<15 } | { 'SSIM':<12 }")
print("="*55)
print(f"{ 'FBP (Baseline)':<20 } | { fbp_psnr:<15.4f } | { fbp_ssim:<12.4f }")
print(f"{ 'TpV (Variational)':<20 } | { tpv_psnr:<15.4f } | { tpv_ssim:<12.4f }")
print(f"{ 'UNet (Deep Learning)':<20 } | { unet_psnr:<15.4f } | { unet_ssim:<12.4f }")
print("="*55)

## 7. Comparative Visual Panel (The Deliverable)

Plot the final 2-row visual comparison panel affiancing Ground Truth, reconstructions under all methods, error maps, and center crops side-by-side. The figure is saved in the `/outputs/comparison/` directory.

In [ ]:
# Convert to 2D numpy arrays
gt_img = x_true_cpu[0, 0].numpy()
fbp_img = x_fbp_norm[0, 0].numpy()
tpv_img = x_tpv_norm[0, 0].numpy()
unet_img = x_unet_norm[0, 0].numpy()

# Compute error maps
fbp_error = np.abs(fbp_img - gt_img)
tpv_error = np.abs(tpv_img - gt_img)
unet_error = np.abs(unet_img - gt_img)
crop_slice = (slice(96, 160), slice(96, 160))

fig, axes = plt.subplots(2, 4, figsize=(15, 8), constrained_layout=True)
fig.suptitle(f"Unified Reconstruction Comparison Panel - {n_views} views - Slice {slice_idx}", fontsize=15)

# First Row: Main images
axes[0, 0].imshow(gt_img, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 0].set_title("Ground Truth")
axes[0, 0].axis("off")

axes[0, 1].imshow(fbp_img, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 1].set_title(f"Noisy FBP Baseline\nPSNR: {fbp_psnr:.2f} dB | SSIM: {fbp_ssim:.4f}")
axes[0, 1].axis("off")

axes[0, 2].imshow(tpv_img, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 2].set_title(f"TpV (Variational)\nPSNR: {tpv_psnr:.2f} dB | SSIM: {tpv_ssim:.4f}")
axes[0, 2].axis("off")

axes[0, 3].imshow(unet_img, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 3].set_title(f"UNet (Deep Learning)\nPSNR: {unet_psnr:.2f} dB | SSIM: {unet_ssim:.4f}")
axes[0, 3].axis("off")

# Second Row: Zoom crop and error maps
axes[1, 0].imshow(gt_img[crop_slice], cmap="gray", vmin=0.0, vmax=1.0)
axes[1, 0].set_title("GT Center Crop")
axes[1, 0].axis("off")

im_fbp_err = axes[1, 1].imshow(fbp_error, cmap="magma")
axes[1, 1].set_title("FBP Error |FBP - GT|")
axes[1, 1].axis("off")
fig.colorbar(im_fbp_err, ax=axes[1, 1], shrink=0.8)

im_tpv_err = axes[1, 2].imshow(tpv_error, cmap="magma")
axes[1, 2].set_title("TpV Error |TpV - GT|")
axes[1, 2].axis("off")
fig.colorbar(im_tpv_err, ax=axes[1, 2], shrink=0.8)

im_unet_err = axes[1, 3].imshow(unet_error, cmap="magma")
axes[1, 3].set_title("UNet Error |UNet - GT|")
axes[1, 3].axis("off")
fig.colorbar(im_unet_err, ax=axes[1, 3], shrink=0.8)

# Save final unified comparison panel under /outputs/comparison/
output_fig_path = OUTPUT_DIR / f"comparison_panel_{n_views}.png"
fig.savefig(output_fig_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved final comparison panel:", output_fig_path)